<a href="https://colab.research.google.com/github/2026-2-Generative-AI-Project/Segmentation-SAM/blob/feature%2Fsam-finetune/labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 정답 마스크 라벨링 (3번 노트북)

`LABELING.md` 규칙대로 그릇마다 번호를 칠해서 `dataset/gt_inst/`에 **번호 PNG + json**으로 저장합니다.

**방식:** 1번 노트북 6번 셀이 저장한 DINO 후보(`seg_pipeline/dino_handoff/candidates.json`)로 **초안**을 깔고, 사람이 클릭으로 고칩니다.
- 초안은 모델 결과라 틀릴 수 있어요. **한 장씩 꼭 눈으로 확인하고 저장**하세요. 그대로 저장하면 모델 실수가 정답이 됩니다.
- 그릇 = 그릇 + 그 위 음식 + 걸친 수저(수저는 따로 번호, 주인 그릇 지정). 쟁반, 컵, 손, 테이블은 칠하지 않음

**순서**
1. 0번, 1번 셀 실행 → 2번 셀(도구) 실행 → 3번 셀에서 라벨링 시작
2. 라벨링이 끝나면 2번 노트북 **1-1번**(라벨 검사) → 1번 노트북 **2번**(`split.csv`에 반영)

**둘이 나눠서 할 때:** 3번 셀에서 `part=(0, 2)`(짝수 번째 사진)와 `part=(1, 2)`(홀수 번째 사진)로 나누면 같은 사진을 겹쳐서 하지 않아요.

# 0. 설치

In [ ]:
!SAM2_BUILD_CUDA=0 pip install -q git+https://github.com/facebookresearch/sam2.git
!pip install -q pillow-heif

# 1. 경로, 설정, 모델

In [ ]:
import os, glob, json, io, base64, html
import numpy as np
import torch
from PIL import Image, ImageOps, ImageDraw, ImageFont
from pillow_heif import register_heif_opener
from scipy import ndimage
import cv2
from IPython.display import display, HTML, JSON
from google.colab import drive, output

register_heif_opener()
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AdCanvas_nukki'
IN_DIR = f'{BASE}/dataset/images'
GT_INST_DIR = f'{BASE}/dataset/gt_inst'
GT_DIR = f'{BASE}/dataset/gt'
CAND_JSON = f'{BASE}/seg_pipeline/dino_handoff/candidates.json'
for d in (GT_INST_DIR, GT_DIR):
    os.makedirs(d, exist_ok=True)

LABELER = ''   # 라벨링하는 사람 이름 (json에 기록, 서로 검수할 때 참고)

LAB = dict(
    max_side=2048,            # 작업 해상도 (1번 노트북과 같음)
    disp_side=1100,           # 화면에 보여 주는 크기 (긴 변)
    sam_model='facebook/sam2.1-hiera-small',
    utensil_words=['spoon', 'chopsticks'],
    utensil_overlap=0.2,      # 수저의 20% 이상이 그릇 외곽 안에 있으면 그 그릇에 걸친 것으로 봄
    mm_score=0.6, mm_inside=0.9,   # 초안: SAM 후보 3개 중 전체 물체 고르기 (1번 노트북 5-1번과 같은 규칙)
    box_margin=0.05,          # 박스 밖에 포함 점을 찍으면 박스를 그만큼 여유 있게 넓힘
)
EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.heic', '.heif')
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')


def stem(p):
    return os.path.splitext(os.path.basename(p))[0]


def load_image(p, max_side=LAB['max_side']):
    img = ImageOps.exif_transpose(Image.open(p)).convert('RGB')
    if max_side:
        img.thumbnail((max_side, max_side), Image.LANCZOS)
    return img


PATHS = {stem(p): p for p in sorted(glob.glob(f'{IN_DIR}/*')) if p.lower().endswith(EXTS)}
CANDS = json.load(open(CAND_JSON))['images'] if os.path.exists(CAND_JSON) else {}

from sam2.sam2_image_predictor import SAM2ImagePredictor
predictor = SAM2ImagePredictor.from_pretrained(LAB['sam_model'], device=device)   # 원본 SAM (초안, 클릭 보정용)

done = sum(os.path.exists(f'{GT_INST_DIR}/{n}.png') for n in PATHS)
print(f'device: {device} | 사진 {len(PATHS)}장 | 라벨 완료 {done}장 | DINO 후보 있는 사진 {len(CANDS)}장')
if not CANDS:
    print('candidates.json이 없음 → 초안 없이 박스로 직접 추가해야 함 (1번 노트북 5-1번, 6번을 먼저 실행하면 초안이 생김)')

# 2. 라벨링 도구
실행만 하면 됩니다 (화면은 3번 셀에서 열림).

**화면 사용법**
| 동작 | 방법 |
|---|---|
| 그릇 고르기 | 아래 목록의 번호 버튼, 또는 `선택` 모드에서 그릇 클릭 |
| 포함 / 제외 | `포함 점` 모드에서 클릭 (Shift+클릭 = 제외 점) |
| 새 그릇, 새 수저 | `박스로 새 그릇` / `박스로 새 수저` 모드 → 두 꼭짓점을 차례로 클릭 |
| 수저의 주인 | 수저를 고른 뒤 `주인:` 옆 번호 클릭 |
| 음식이 비어 있을 때 | `구멍 채우기` |
| 저장 | `저장` 또는 `저장하고 다음` (저장하지 않고 넘어가면 수정 내용이 사라짐) |

화면의 번호가 저장되는 번호예요 (큰 그릇부터, 수저는 맨 뒤).

In [ ]:
COLORS = np.array([(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48), (145, 30, 180),
                   (70, 240, 240), (240, 50, 230), (210, 245, 60), (250, 190, 212), (0, 128, 128),
                   (220, 190, 255), (170, 110, 40), (128, 0, 0), (170, 255, 195), (128, 128, 0)])


def mask_box(m):
    ys, xs = np.nonzero(m)
    return [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]


def box_mask(shape, box, expand=0.1):
    x0, y0, x1, y1 = box
    ex, ey = expand * (x1 - x0), expand * (y1 - y0)
    out = np.zeros(shape, bool)
    out[max(int(y0 - ey), 0):max(int(np.ceil(y1 + ey)), 0), max(int(x0 - ex), 0):max(int(np.ceil(x1 + ex)), 0)] = True
    return out


def crop_apply(fn, m, pad=4):
    out = np.zeros_like(m)
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return out
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1])
    return out


def fill_close(m):
    return crop_apply(lambda c: ndimage.binary_fill_holes(ndimage.binary_closing(c, iterations=3)), m)


def hull(m):
    def f(c):
        pts = cv2.findNonZero(c.astype(np.uint8))
        out = np.zeros(c.shape, np.uint8)
        cv2.fillConvexPoly(out, cv2.convexHull(pts), 1)
        return out.astype(bool)
    return crop_apply(f, m, pad=0)


def pick_whole(ms, ss, box):
    ok = [j for j in range(len(ms)) if ms[j].any() and ss[j] >= LAB['mm_score'] * ss.max()]
    bm = box_mask(ms[0].shape, box)
    ok = [j for j in ok if (ms[j] & bm).sum() >= LAB['mm_inside'] * ms[j].sum()]
    return ms[max(ok, key=lambda j: ms[j].sum())] if ok else ms[int(ss.argmax())]


class Labeler:
    def __init__(self, names):
        self.names, self.uid = names, 0
        self.load(0)

    # ---------- 사진 불러오기 ----------
    def load(self, i):
        self.i = i % len(self.names)
        self.name = self.names[self.i]
        self.img = load_image(PATHS[self.name])
        self.np = np.array(self.img)
        self.H, self.W = self.np.shape[:2]
        self.ds = min(LAB['disp_side'] / max(self.W, self.H), 1.0)
        self.disp_size = (round(self.W * self.ds), round(self.H * self.ds))
        self.disp_base = self.img.resize(self.disp_size, Image.BILINEAR)
        predictor.set_image(self.np)
        self.mode, self.corner, self.msg, self.dirty = 'point', None, '', False
        self.saved = os.path.exists(f'{GT_INST_DIR}/{self.name}.png')
        self.objs = self.from_saved() if self.saved else self.prelabel()
        self.cur = self.ordered()[0]['uid'] if self.objs else None

    def new_obj(self, kind, box, mask=None):
        self.uid += 1
        box = [max(0.0, box[0]), max(0.0, box[1]), min(float(self.W), box[2]), min(float(self.H), box[3])]
        if mask is None:
            m, s, _ = predictor.predict(box=np.array(box, np.float32), multimask_output=True)
            m, s = np.asarray(m) > 0.5, np.asarray(s)
            mask = fill_close(pick_whole(m, s, box)) if kind == 'dish' else m[int(s.argmax())]
        return {'uid': self.uid, 'type': kind, 'parent': None, 'box': box,
                'points': [], 'labels': [], 'base': mask, 'mask': mask}

    def prelabel(self):
        rec = CANDS.get(self.name)
        if not rec:
            self.msg = '이 사진은 DINO 후보가 없어요. 박스로 직접 추가하세요.'
            return []
        sx, sy = self.W / rec['size'][0], self.H / rec['size'][1]
        boxes = lambda c: [c['box'][0] * sx, c['box'][1] * sy, c['box'][2] * sx, c['box'][3] * sy]
        objs = [self.new_obj('dish', boxes(c)) for c in rec['candidates'] if c['dish']]
        for c in rec['candidates']:
            if c['label'] in LAB['utensil_words']:
                u = self.new_obj('utensil', boxes(c))
                u['parent'] = self.guess_parent(u, objs)
                if u['parent'] is not None and u['mask'].any():   # 그릇에 안 걸친 수저는 라벨 대상 아님
                    objs.append(u)
        objs = [o for o in objs if o['mask'].any()]
        self.msg = f'초안: 그릇 {sum(o["type"] == "dish" for o in objs)}개, 수저 {sum(o["type"] == "utensil" for o in objs)}개. 확인하고 고친 뒤 저장하세요.'
        return objs

    def from_saved(self):
        arr = np.asarray(Image.open(f'{GT_INST_DIR}/{self.name}.png').resize((self.W, self.H), Image.NEAREST))
        meta = json.load(open(f'{GT_INST_DIR}/{self.name}.json'))
        idmap, objs = {}, []
        for o in meta['objects']:
            m = arr == int(o['id'])
            if not m.any():
                continue
            ob = self.new_obj(o['type'], mask_box(m), mask=m)
            idmap[int(o['id'])] = ob['uid']
            objs.append(ob)
        for o, ob in zip([o for o in meta['objects'] if (arr == int(o['id'])).any()], objs):
            if o['type'] == 'utensil':
                ob['parent'] = idmap.get(int(o.get('parent', -1)))
        self.msg = '저장된 라벨을 불러왔어요 (수정 모드).'
        return objs

    def guess_parent(self, u, dishes):
        best, score = None, 0
        for d in dishes:
            if d['type'] != 'dish' or not d['mask'].any():
                continue
            ov = (u['mask'] & hull(d['mask'])).sum()
            if ov > score:
                best, score = d['uid'], ov
        return best if score > LAB['utensil_overlap'] * max(u['mask'].sum(), 1) else None

    # ---------- 편집 ----------
    def get(self, uid):
        return next((o for o in self.objs if o['uid'] == uid), None)

    def ordered(self):
        """저장 순서 = 화면 번호: 큰 그릇부터(작은 그릇이 위에 칠해짐), 수저는 맨 뒤"""
        dishes = sorted([o for o in self.objs if o['type'] == 'dish'], key=lambda o: -o['mask'].sum())
        return dishes + [o for o in self.objs if o['type'] == 'utensil']

    def refine(self, o):
        if not o['points']:
            o['mask'] = o['base']
            return
        m, _, _ = predictor.predict(point_coords=np.array(o['points'], np.float32),
                                    point_labels=np.array(o['labels'], np.int32),
                                    box=np.array(o['box'], np.float32), multimask_output=False)
        o['mask'] = np.asarray(m)[0] > 0.5

    def click(self, x, y, shift):
        X, Y = min(max(x / self.ds, 0), self.W - 1), min(max(y / self.ds, 0), self.H - 1)
        if self.mode in ('box_dish', 'box_utensil'):
            if self.corner is None:
                self.corner, self.msg = (X, Y), '반대쪽 꼭짓점을 클릭하세요.'
                return
            (a, b), self.corner = self.corner, None
            box = [min(a, X), min(b, Y), max(a, X), max(b, Y)]
            if box[2] - box[0] < 5 or box[3] - box[1] < 5:
                self.msg = '박스가 너무 작아요. 다시 그려 주세요.'
                return
            kind = 'dish' if self.mode == 'box_dish' else 'utensil'
            o = self.new_obj(kind, box)
            if kind == 'utensil':
                o['parent'] = self.guess_parent(o, self.objs)
            self.objs.append(o)
            self.cur, self.mode, self.dirty = o['uid'], 'point', True
            self.msg = f'{"그릇을" if kind == "dish" else "수저를"} 추가했어요. 포함/제외 점으로 다듬으세요.'
            if kind == 'utensil' and o['parent'] is None:
                self.msg += ' 주인 그릇을 직접 골라 주세요.'
            return
        if self.mode == 'select':
            hit = [o for o in self.ordered() if o['mask'][int(Y), int(X)]]
            if hit:
                self.cur = hit[-1]['uid']
                self.msg = ''
            return
        o = self.get(self.cur)
        if o is None:
            self.msg = '먼저 그릇을 고르거나 박스로 추가하세요.'
            return
        lab = 0 if (shift or self.mode == 'neg') else 1
        o['points'].append([X, Y]); o['labels'].append(lab)
        if lab == 1:   # 박스 밖에 포함 점을 찍으면 박스를 넓힘
            bx = o['box']; mw, mh = LAB['box_margin'] * (bx[2] - bx[0]), LAB['box_margin'] * (bx[3] - bx[1])
            o['box'] = [max(0.0, min(bx[0], X - mw)), max(0.0, min(bx[1], Y - mh)),
                        min(float(self.W), max(bx[2], X + mw)), min(float(self.H), max(bx[3], Y + mh))]
        self.refine(o)
        self.dirty, self.msg = True, ''

    def act(self, a, args):
        o = self.get(self.cur)
        if a == 'click':
            self.click(*args)
        elif a == 'mode':
            self.mode, self.corner = args[0], None
            self.msg = {'box_dish': '그릇의 두 꼭짓점을 차례로 클릭하세요.', 'box_utensil': '수저의 두 꼭짓점을 차례로 클릭하세요.',
                        'select': '고를 그릇을 클릭하세요.'}.get(args[0], '')
        elif a == 'select':
            self.cur, self.msg = args[0], ''
        elif a == 'parent' and o and o['type'] == 'utensil':
            o['parent'], self.dirty = args[0], True
        elif a == 'fill' and o:
            o['mask'] = o['base'] = fill_close(o['mask']); o['points'], o['labels'] = [], []
            self.dirty = True
        elif a == 'undo' and o and o['points']:
            o['points'].pop(); o['labels'].pop(); self.refine(o); self.dirty = True
        elif a == 'clear' and o:
            o['points'], o['labels'] = [], []; self.refine(o); self.dirty = True
        elif a == 'retype' and o:
            o['type'] = 'utensil' if o['type'] == 'dish' else 'dish'
            o['parent'] = self.guess_parent(o, self.objs) if o['type'] == 'utensil' else None
            for u in self.objs:
                if u['parent'] == o['uid'] and o['type'] == 'utensil':
                    u['parent'] = None
            self.dirty = True
        elif a == 'delete' and o:
            self.objs.remove(o)
            for u in self.objs:
                if u['parent'] == o['uid']:
                    u['parent'] = None
            self.cur = self.ordered()[0]['uid'] if self.objs else None
            self.dirty = True
        elif a == 'save':
            self.save()
        elif a == 'save_next':
            if self.save():
                self.load(self.i + 1)
        elif a == 'next':
            self.load(self.i + 1)
        elif a == 'prev':
            self.load(self.i - 1)
        elif a == 'redo_prelabel':
            self.objs = self.prelabel()
            self.cur = self.ordered()[0]['uid'] if self.objs else None
            self.dirty = True

    def save(self):
        order = [o for o in self.ordered() if o['mask'].any()]
        dish_uids = {o['uid'] for o in order if o['type'] == 'dish'}
        bad = [o for o in order if o['type'] == 'utensil' and o['parent'] not in dish_uids]
        if bad:
            self.msg = '저장 안 됨: 주인 그릇이 없는 수저가 있어요 (빨간 글씨). 주인을 고르거나 삭제하세요.'
            return False
        arr = np.zeros((self.H, self.W), np.uint8)
        idmap = {}
        for k, o in enumerate(order, 1):
            arr[o['mask']] = k
            idmap[o['uid']] = k
        objects = []
        for o in order:
            k = idmap[o['uid']]
            if not (arr == k).any():
                continue   # 다른 물체에 완전히 가려짐
            ob = {'id': k, 'type': o['type']}
            if o['type'] == 'utensil':
                ob['parent'] = idmap[o['parent']]
            objects.append(ob)
        present = {ob['id'] for ob in objects}
        if any(ob['type'] == 'utensil' and ob['parent'] not in present for ob in objects):
            self.msg = '저장 안 됨: 수저의 주인 그릇이 다른 물체에 완전히 가려졌어요.'
            return False
        Image.fromarray(arr).save(f'{GT_INST_DIR}/{self.name}.png')
        json.dump({'objects': objects, 'labeler': LABELER, 'note': ''},
                  open(f'{GT_INST_DIR}/{self.name}.json', 'w'), ensure_ascii=False, indent=1)
        Image.fromarray(((arr > 0) * 255).astype(np.uint8)).save(f'{GT_DIR}/{self.name}.png')   # 1번 노트북용 흑백 정답
        self.saved, self.dirty = True, False
        self.msg = f'저장했어요: 그릇 {sum(o["type"] == "dish" for o in objects)}개, 수저 {sum(o["type"] == "utensil" for o in objects)}개'
        return True

    # ---------- 화면 ----------
    def render(self):
        img = np.array(self.disp_base).astype(np.float32)
        order = self.ordered()
        dm = {o['uid']: np.asarray(Image.fromarray(o['mask']).resize(self.disp_size, Image.NEAREST)) for o in order}
        for k, o in enumerate(order):
            m = dm[o['uid']]
            img[m] = img[m] * 0.5 + COLORS[k % len(COLORS)] * 0.5
        out = Image.fromarray(img.clip(0, 255).astype(np.uint8))
        d = ImageDraw.Draw(out)
        try:
            font = ImageFont.load_default(size=18)
        except TypeError:
            font = ImageFont.load_default()
        cur = self.get(self.cur)
        if cur is not None and cur['uid'] in dm:
            m = dm[cur['uid']]
            edge = m ^ ndimage.binary_erosion(m, iterations=2)
            arr = np.array(out); arr[edge] = (255, 255, 0); out = Image.fromarray(arr); d = ImageDraw.Draw(out)
            x0, y0, x1, y1 = [v * self.ds for v in cur['box']]
            d.rectangle([x0, y0, x1, y1], outline=(255, 255, 0), width=1)
            for (px, py), lb in zip(cur['points'], cur['labels']):
                px, py = px * self.ds, py * self.ds
                d.ellipse([px - 6, py - 6, px + 6, py + 6], fill=(0, 220, 0) if lb else (230, 0, 0), outline=(255, 255, 255), width=2)
        for k, o in enumerate(order, 1):
            m = dm[o['uid']]
            if m.any():
                ys, xs = np.nonzero(m)
                cx, cy = float(np.median(xs)), float(np.median(ys))
                t = f'{k}' if o['type'] == 'dish' else f'{k}s'
                d.text((cx, cy), t, fill=(255, 255, 255), font=font, stroke_width=3, stroke_fill=(0, 0, 0))
        if self.corner is not None:
            cx, cy = self.corner[0] * self.ds, self.corner[1] * self.ds
            d.line([cx - 10, cy, cx + 10, cy], fill=(255, 255, 0), width=3)
            d.line([cx, cy - 10, cx, cy + 10], fill=(255, 255, 0), width=3)
        buf = io.BytesIO()
        out.save(buf, 'JPEG', quality=85)
        return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode()

    def info(self):
        def btn(label, a, args=(), on=False, color=None):
            style = 'margin:2px;padding:4px 9px;border-radius:6px;border:1px solid #888;cursor:pointer;'
            style += 'font-weight:bold;border:2px solid #111;' if on else ''
            style += f'background:{color};' if color else ''
            return f'<button style="{style}" onclick=\'labAct("{a}", {json.dumps(list(args))})\'>{html.escape(label)}</button>'
        order = self.ordered()
        num = {o['uid']: k for k, o in enumerate(order, 1)}
        done = sum(os.path.exists(f'{GT_INST_DIR}/{n}.png') for n in self.names)
        state = '저장됨' if self.saved and not self.dirty else ('수정 중 (저장 안 됨)' if self.dirty else '아직 저장 안 됨')
        h = [f'<div style="font-size:16px;margin:4px 0"><b>{html.escape(self.name)}</b> ({self.i + 1}/{len(self.names)}, 완료 {done}) | <b>{state}</b></div>']
        h.append('<div>모드: ' + ''.join(btn(l, 'mode', [m], self.mode == m) for l, m in
                 [('포함 점', 'point'), ('제외 점', 'neg'), ('선택', 'select'), ('박스로 새 그릇', 'box_dish'), ('박스로 새 수저', 'box_utensil')])
                 + ' <span style="color:#666">Shift+클릭 = 제외 점</span></div>')
        chips = []
        for k, o in enumerate(order):
            c = 'rgb(%d,%d,%d)' % tuple(COLORS[k % len(COLORS)])
            lab = f'{k + 1} 그릇' if o['type'] == 'dish' else f'{k + 1} 수저→{num.get(o["parent"], "?")}'
            chips.append(btn(lab, 'select', [o['uid']], o['uid'] == self.cur, c))
        h.append('<div>물체: ' + (''.join(chips) if chips else '<span style="color:#666">없음</span>') + '</div>')
        o = self.get(self.cur)
        if o is not None:
            row = '선택한 물체: ' + btn('구멍 채우기', 'fill') + btn('점 하나 취소', 'undo') + btn('점 모두 지우기', 'clear') \
                + btn('그릇↔수저', 'retype') + btn('삭제', 'delete', color='#f4c7c3')
            if o['type'] == 'utensil':
                warn = '' if o['parent'] in num and self.get(o['parent'])['type'] == 'dish' else ' <b style="color:#c00">주인 없음</b>'
                row += ' | 주인: ' + ''.join(btn(str(num[d['uid']]), 'parent', [d['uid']], d['uid'] == o['parent'])
                                            for d in order if d['type'] == 'dish') + warn
            h.append(f'<div>{row}</div>')
        h.append('<div style="margin-top:4px">' + btn('이전', 'prev') + btn('저장', 'save', color='#c8e6c9')
                 + btn('저장하고 다음', 'save_next', color='#a5d6a7') + btn('다음 (저장 안 함)', 'next')
                 + btn('초안 다시 만들기', 'redo_prelabel') + '</div>')
        if self.msg:
            h.append(f'<div style="margin-top:4px;color:#0b57d0">{html.escape(self.msg)}</div>')
        return ''.join(h)

    def payload(self):
        return {'img': self.render(), 'info': self.info()}


def _act(a, args):
    try:
        lab.act(a, list(args))
    except Exception as e:
        lab.msg = f'에러: {type(e).__name__}: {e}'
    return JSON(lab.payload())


output.register_callback('lab.act', _act)

PAGE = """
<div id="labinfo" style="font-family:sans-serif;font-size:14px">__INFO__</div>
<img id="labimg" src="__IMG__" style="max-width:100%;cursor:crosshair;margin-top:6px" onclick="labClick(event, this)">
<script>
async function labAct(a, args) {
  document.getElementById('labinfo').style.opacity = 0.5;
  const r = await google.colab.kernel.invokeFunction('lab.act', [a, args], {});
  const d = r.data['application/json'];
  document.getElementById('labimg').src = d.img;
  document.getElementById('labinfo').innerHTML = d.info;
  document.getElementById('labinfo').style.opacity = 1;
}
function labClick(e, el) {
  const r = el.getBoundingClientRect();
  labAct('click', [(e.clientX - r.left) * el.naturalWidth / r.width, (e.clientY - r.top) * el.naturalHeight / r.height, e.shiftKey]);
}
</script>
"""


def start(only=None, part=None, include_done=False):
    """only: 이 사진들만 / part=(0, 2) 짝수 번째, (1, 2) 홀수 번째 / include_done=True면 이미 저장한 사진도 (수정용)"""
    global lab
    names = sorted(PATHS)
    if only:
        names = [n for n in names if n in set(only)]
    elif not include_done:
        names = [n for n in names if not os.path.exists(f'{GT_INST_DIR}/{n}.png')]
    if part:
        names = names[part[0]::part[1]]
    if not names:
        print('라벨링할 사진이 없어요.')
        return
    lab = Labeler(names)
    p = lab.payload()
    display(HTML(PAGE.replace('__INFO__', p['info']).replace('__IMG__', p['img'])))

# 3. 라벨링 시작
* 아직 라벨 안 한 사진만 차례로 나옴
* 나눠서 할 때: `start(part=(0, 2))` / `start(part=(1, 2))`
* 특정 사진만: `start(only=['IMG_2184', 'IMG_6594'])`
* 저장한 사진 다시 고치기: `start(only=['IMG_2184'])` 또는 `start(include_done=True)`
* 화면이 멈추거나 이상하면 이 셀을 다시 실행

In [ ]:
start()

# 4. 라벨링 현황

In [ ]:
names = sorted(PATHS)
done = [n for n in names if os.path.exists(f'{GT_INST_DIR}/{n}.png')]
print(f'완료 {len(done)} / {len(names)}')
for n in done:
    j = json.load(open(f'{GT_INST_DIR}/{n}.json'))
    ds = sum(o['type'] == 'dish' for o in j['objects']); us = sum(o['type'] == 'utensil' for o in j['objects'])
    print(f'  {n}: 그릇 {ds}, 수저 {us}' + (f" ({j.get('labeler')})" if j.get('labeler') else ''))
print('남은 사진:', [n for n in names if n not in set(done)])